In [0]:
-- ============================================
-- PROJECT: Sales & Customer Analytics
-- LAYER: GOLD (Business-Level Analysis)
-- PURPOSE: KPI generation & dashboard support
-- ============================================

USE workspace.gold;

-- ============================================
-- 1. DATA EXPLORATION
-- ============================================

SHOW TABLES;

-- Understanding the Table Structures
DESCRIBE TABLE dim_customers;
DESCRIBE TABLE fact_sales;
DESCRIBE TABLE dim_products;

-- ============================================
-- 2. MARKET UNDERSTANDING
-- ============================================
-- Countries contributing to actual sales 

SELECT DISTINCT 
  c.country
FROM fact_sales f
JOIN dim_customers c 
  ON f.customer_key = c.customer_key
ORDER BY c.country;

-- ============================================
-- 3. CORE KPIs
-- ============================================

SELECT
  SUM(sales_amount) AS total_sales,
  SUM(quantity) AS total_quantity,
  COUNT(DISTINCT customer_key) AS total_customers,
  COUNT(DISTINCT order_number) AS total_orders
FROM fact_sales;

-- ============================================
-- 4. PRODUCT HIERARCHY
-- ============================================
-- Retrieve a list of unique categories, subcategories, and products

SELECT DISTINCT 
  category, 
  subcategory, 
  product_name
FROM dim_products
ORDER BY category, subcategory, product_name;

-- ============================================
-- 5. ORDERS BY CATEGORY
-- ============================================
-- Find the total number of orders by categories

SELECT
  p.category,
  COUNT(DISTINCT f.order_number) AS total_orders
FROM fact_sales f 
LEFT JOIN dim_products p 
  ON f.product_key = p.product_id
WHERE p.category IS NOT NULL
GROUP BY p.category
ORDER BY total_orders DESC;

-- ============================================
-- 6. TREND ANALYSIS (COMBINED - BETTER DESIGN)
-- ============================================
-- Orders over time

SELECT
  date_trunc('month', order_date) AS order_month,
  COUNT(order_number) AS Total_Orders,
  SUM(sales_amount) AS total_sales
FROM fact_sales
WHERE date_trunc('month', order_date) IS NOT NULL
GROUP BY order_month
ORDER BY order_month;

-- ============================================
-- 7. CATEGORY PERFORMANCE
-- ============================================

SELECT
  p.category,
  SUM(f.sales_amount) AS total_sales
FROM fact_sales f 
JOIN dim_products p 
  ON f.product_key = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC;

-- ============================================
-- 8. CUSTOMER ANALYSIS (GEOGRAPHY)
-- ============================================
-- Customer demographics

SELECT
  c.country,
  COUNT(DISTINCT c.customer_key) AS total_customers
FROM fact_sales f 
JOIN dim_customers c 
  ON f.customer_key = c.customer_key
GROUP BY c.country
ORDER BY total_customers DESC;

-- ============================================
-- 9. PRODUCT PERFORMANCE
-- ============================================
-- Product Popularity
-- Top-selling products

SELECT
  p.product_name,
  SUM(f.sales_amount) AS total_sales
FROM fact_sales f 
JOIN dim_products p 
  ON f.product_key = p.product_id
GROUP BY p.product_name
ORDER BY total_sales DESC;

-- ============================================
-- 10. CUSTOMER SEGMENTATION
-- ============================================

WITH customer_sales AS (
  SELECT
    customer_key,
    SUM(sales_amount) AS total_spent
  FROM fact_sales
  GROUP BY customer_key
)

SELECT
  customer_key,
  total_spent,
  CASE
    WHEN total_spent > 10000 THEN 'VIP'
    WHEN total_spent > 5000 THEN 'Regular'
    ELSE 'New'
  END AS customer_segment
FROM customer_sales;

-- ============================================
-- 11. CATEGORY CONTRIBUTION (%)
-- ============================================

SELECT
  p.category,
  SUM(f.sales_amount) AS total_sales,
  ROUND(
    100 * SUM(f.sales_amount) / SUM(SUM(f.sales_amount)) OVER (), 
    2
  ) AS contribution_pct
FROM fact_sales f
JOIN dim_products p 
  ON f.product_key = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC;



Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.